## Vývoj mezd v ČR 2011–2024

Stažení dat:

In [ ]:
import os
import requests
import re

# roky, které chceme stáhnout (bez 2025)
YEARS = list(range(2011, 2025))

BASE_DOMAIN = "https://www.ispv.cz"
BASE_ARCHIVE_URL = "https://www.ispv.cz/cz/Vysledky-setreni/Archiv/{year}.aspx"
AKTUALNI_URL = "https://www.ispv.cz/cz/Vysledky-setreni/Aktualni.aspx"

# hlavní složka
os.makedirs("ispv_data", exist_ok=True)

# Session pro zachování spojení a správných hlaviček prohlížeče
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})

def table_code(year):
    """Vrátí kód tabulky podle vzorce 'rok4'."""
    return (year % 100) * 10 + 4  # např. 2018 → 184

for year in YEARS:
    code = table_code(year)
    year_folder = f"ispv_data/{year}"
    os.makedirs(year_folder, exist_ok=True)
    
    print(f"\n=== Rok {year} (kód {code}) ===")
    
    # Zkusíme nejprve stránku archivu
    url = BASE_ARCHIVE_URL.format(year=year)
    r = session.get(url)
    
    # Pokud pro daný rok není v archivu (např. nejnovější rok může být jinde), zkusíme sekci "Aktuální"
    if r.status_code != 200 or "Nenalezeno" in r.text or "Chyba" in r.text:
        url = AKTUALNI_URL
        r = session.get(url)
        
    # Pomocí regulárního výrazu najdeme všechny reálné odkazy ke stažení Excelů
    # Hledáme soubory s koncovkou .xlsx, které obsahují správný kód roku
    pattern = r'<a[^>]+href=["\']([^"\']+)["\'][^>]*>([A-Za-z]+_' + str(code) + r'_(?:mzs|pls|MZS|PLS)\.xlsx?)</a>'
    matches = re.findall(pattern, r.text)
    
    # Odstraníme možné duplicity z HTML
    unique_files = {}
    for href, filename in matches:
        unique_files[filename] = href
        
    if not unique_files:
        print(f"  ❌ Žádné odkazy pro kód {code} nebyly nalezeny na {url}.")
        continue
        
    for filename, href in unique_files.items():
        # Zajištění kompletní URL
        if not href.startswith("http"):
            href = BASE_DOMAIN + href
            
        path = f"{year_folder}/{filename}"
        print(f"Stahuju: {filename}")
        
        try:
            file_r = session.get(href)
            if file_r.status_code == 200:
                # Kontrola, zda jsme omylem zase nestáhli chybovou stránku
                if b"<html" not in file_r.content[:20].lower():
                    with open(path, "wb") as f:
                        f.write(file_r.content)
                else:
                    print(f"  ❌ Stažen HTML kód místo Excelu ({filename})")
            else:
                print(f"  ❌ Chyba serveru při stahování ({href})")
        except Exception as e:
            print(f"  ❌ Výjimka při stahování: {e}")



=== Rok 2011 (kód 114) ===
Stahuju: CR_114_MZS.xls
Stahuju: Pra_114_mzs.xls
Stahuju: Str_114_mzs.xls
Stahuju: Jic_114_mzs.xls
Stahuju: Plz_114_mzs.xls
Stahuju: Kar_114_mzs.xls
Stahuju: Ust_114_mzs.xls
Stahuju: Lib_114_mzs.xls
Stahuju: Kra_114_mzs.xls
Stahuju: Par_114_mzs.xls
Stahuju: Vys_114_mzs.xls
Stahuju: Jim_114_mzs.xls
Stahuju: Olo_114_mzs.xls
Stahuju: Zli_114_mzs.xls
Stahuju: Mos_114_mzs.xls
Stahuju: CR_114_PLS.xls
Stahuju: Pra_114_pls.xls
Stahuju: Str_114_pls.xls
Stahuju: Jic_114_pls.xls
Stahuju: Plz_114_pls.xls
Stahuju: Kar_114_pls.xls
Stahuju: Ust_114_pls.xls
Stahuju: Lib_114_pls.xls
Stahuju: Kra_114_pls.xls
Stahuju: Par_114_pls.xls
Stahuju: Vys_114_pls.xls
Stahuju: Jim_114_pls.xls
Stahuju: Olo_114_pls.xls
Stahuju: Zli_114_pls.xls
Stahuju: Mos_114_pls.xls

=== Rok 2012 (kód 124) ===
Stahuju: CR_124_MZS.xls
Stahuju: Pra_124_mzs.xls
Stahuju: Str_124_mzs.xls
Stahuju: Jic_124_mzs.xls
Stahuju: Plz_124_mzs.xls
Stahuju: Kar_124_mzs.xls
Stahuju: Ust_124_mzs.xls
Stahuju: Lib_124_mzs.x

Instalace knihovny, která umožní číst staré Excely (.xls):

In [4]:
pip install xlrd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Ověření struktury dat:

In [8]:
import os

BASE_FOLDER = r"C:\Users\marie\Documents\Datová analytika - Python, Big data, ML\Projekt\ispv_data"

for file in os.listdir(BASE_FOLDER):
    print(file)


2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


Extrakce metadat z názvů souborů:

In [9]:
for file in os.listdir(BASE_FOLDER):
    if file.endswith(".xlsx"):
        print(file, extract_metadata_from_filename(file))


Parser:

In [1]:
import os
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')
BASE_FOLDER = "ispv_data"
REGION_MAP = {
    "Pra": "Hlavní město Praha", "Str": "Středočeský kraj",
    "Jhc": "Jihočeský kraj", "Jic": "Jihočeský kraj",
    "Plz": "Plzeňský kraj", "Kvk": "Karlovarský kraj", "Kar": "Karlovarský kraj",
    "Ust": "Ústecký kraj", "Lib": "Liberecký kraj",
    "Khr": "Královéhradecký kraj", "Kra": "Královéhradecký kraj",
    "Pha": "Pardubický kraj", "Par": "Pardubický kraj",
    "Vys": "Kraj Vysočina", "Jhm": "Jihomoravský kraj", "Jim": "Jihomoravský kraj",
    "Olm": "Olomoucký kraj", "Olo": "Olomoucký kraj",
    "Zln": "Zlínský kraj", "Zli": "Zlínský kraj",
    "Mor": "Moravskoslezský kraj", "Mos": "Moravskoslezský kraj",
    "CR": "Česká republika", "Cr": "Česká republika"
}
def detect_header_rows(df):
    for i in range(min(20, len(df))):
        row_text = " ".join([str(x).lower() for x in df.iloc[i].values])
        if "kč/hod" in row_text or "kč/měs" in row_text or "tis. osob" in row_text:
            return max(0, i - 4), i
    for i in range(1, 15):
        if df.iloc[i].astype(str).str.contains(r"\d").any():
            return max(0, i - 2), i - 1
    return 0, 1
def extract_metadata(filename):
    match = re.match(r"([A-Za-z]+)_(\d{3})_(mzs|pls)\.xlsx?", filename, re.IGNORECASE)
    if match:
        kraj_code, code, sfera = match.groups()
        kraj = REGION_MAP.get(kraj_code.capitalize(), kraj_code)
        return kraj, sfera.lower()
    return None, None
def normalize_column_name(col_name):
    """Převede různorodé názvy sloupců z různých let do jednotného formátu (2011)"""
    nc = col_name.lower()
    
    if "počet zaměstnanců" in nc and "tis. osob" in nc:
        return "Počet zaměstnanců přepočtený podle placených měsíců_tis. osob"
        
    # Normalizace pro Mzdy
    if "hrubá měsíční mzda" in nc:
        if "medián" in nc: return "Hrubá měsíční mzda_Medián_Kč/měs"
        if "index" in nc or "změna" in nc: return "Hrubá měsíční mzda_Meziroční index_%"
        if "průměr" in nc: return "Hrubá měsíční mzda_Průměr_Kč/měs"
        
    # Normalizace pro Platy
    if "hrubý měsíční plat" in nc:
        if "medián" in nc: return "Hrubý měsíční plat_Medián_Kč/měs"
        if "index" in nc or "změna" in nc: return "Hrubý měsíční plat_Meziroční index_%"
        if "průměr" in nc: return "Hrubý měsíční plat_Průměr_Kč/měs"
        
    # Normalizace pro Diferenciace
    if "diferenciace" in nc:
        prefix = "Diferenciace hrubé měsíční mzdy" if "mzdy" in nc else "Diferenciace hrubého měsíčního platu"
        if "1. decil" in nc: return f"{prefix}_1. decil_Kč/měs"
        if "1. kvartil" in nc: return f"{prefix}_1. kvartil_Kč/měs"
        if "3. kvartil" in nc: return f"{prefix}_3. kvartil_Kč/měs"
        if "9. decil" in nc: return f"{prefix}_9. decil_Kč/měs"
    return col_name
pozadovane_sloupce_vek = [
    'Rok', 'Sféra', 'Pohlaví', 'Věk', 'Kraj',
    'Počet zaměstnanců přepočtený podle placených měsíců_tis. osob',
    'Hrubá měsíční mzda_Medián_Kč/měs', 'Hrubá měsíční mzda_Meziroční index_%',
    'Diferenciace hrubé měsíční mzdy_1. decil_Kč/měs', 'Diferenciace hrubé měsíční mzdy_1. kvartil_Kč/měs',
    'Diferenciace hrubé měsíční mzdy_3. kvartil_Kč/měs', 'Diferenciace hrubé měsíční mzdy_9. decil_Kč/měs',
    'Hrubá měsíční mzda_Průměr_Kč/měs', 'Hrubý měsíční plat_Medián_Kč/měs',
    'Hrubý měsíční plat_Meziroční index_%', 'Diferenciace hrubého měsíčního platu_1. decil_Kč/měs',
    'Diferenciace hrubého měsíčního platu_1. kvartil_Kč/měs', 'Diferenciace hrubého měsíčního platu_3. kvartil_Kč/měs',
    'Diferenciace hrubého měsíčního platu_9. decil_Kč/měs', 'Hrubý měsíční plat_Průměr_Kč/měs'
]
pozadovane_sloupce_vzdelani = [
    'Rok', 'Sféra', 'Vzdělání', 'Kraj',
    'Počet zaměstnanců přepočtený podle placených měsíců_tis. osob',
    
    # Sloupce pro mzdovou sféru (MZS)
    'Hrubá měsíční mzda_Medián_Kč/měs', 'Hrubá měsíční mzda_Meziroční index_%',
    'Diferenciace hrubé měsíční mzdy_1. decil_Kč/měs', 'Diferenciace hrubé měsíční mzdy_1. kvartil_Kč/měs',
    'Diferenciace hrubé měsíční mzdy_3. kvartil_Kč/měs', 'Diferenciace hrubé měsíční mzdy_9. decil_Kč/měs',
    'Hrubá měsíční mzda_Průměr_Kč/měs',
    
    # Sloupce pro platovou sféru (PLS) - tyto tady původně chyběly
    'Hrubý měsíční plat_Medián_Kč/měs', 'Hrubý měsíční plat_Meziroční index_%', 
    'Diferenciace hrubého měsíčního platu_1. decil_Kč/měs', 'Diferenciace hrubého měsíčního platu_1. kvartil_Kč/měs', 
    'Diferenciace hrubého měsíčního platu_3. kvartil_Kč/měs', 'Diferenciace hrubého měsíčního platu_9. decil_Kč/měs', 
    'Hrubý měsíční plat_Průměr_Kč/měs'
]
def parse_excel_file(path, year, kraj, sfera):
    try:
        df_dict = pd.read_excel(path, sheet_name=None, header=None)
    except Exception as e:
        print(f"Chyba při čtení {path}: {e}")
        return [], []
    vek_rows = []
    vzdelani_rows = []
    
    for sheet, df in df_dict.items():
        if df.empty: continue
            
        title = ""
        for i in range(min(15, len(df))):
            row_str = " ".join(df.iloc[i].astype(str).str.lower().fillna(''))
            if 'nan' != row_str.strip():
                title += " " + row_str.replace('nan', '').strip()
                
        je_vek = False
        je_vzdelani = False
        
        if ("odpracovaná" not in title) and ("odpracované" not in title):
            if "pohlaví a věku" in title or "pohlaví/věková skupina" in title: je_vek = True
            elif "vzdělání" in title or "vzdelani" in title: je_vzdelani = True
                
        if not je_vek and not je_vzdelani: continue
            
        start_idx, unit_idx = detect_header_rows(df)
        header_block = df.iloc[start_idx:unit_idx+1].ffill(axis=1)
        
        columns = []
        for c in range(len(df.columns)):
            col_parts = []
            for r in range(start_idx, unit_idx + 1):
                val = str(header_block.iloc[r - start_idx, c])
                if val != 'nan' and val.strip() and val != 'NaN':
                    clean_val = re.sub(r'\s+', ' ', val.strip())
                    if not col_parts or col_parts[-1] != clean_val:
                        col_parts.append(clean_val)
            col_name = "_".join(col_parts)
            if not col_name: col_name = f"Sloupec_{c}"
            
            columns.append(normalize_column_name(col_name))
            
        final_columns = []
        seen = set()
        for col in columns:
            new_col = col
            counter = 1
            while new_col in seen:
                new_col = f"{col}_{counter}"
                counter += 1
            seen.add(new_col)
            final_columns.append(new_col)
            
        data_df = df.iloc[unit_idx+1:].copy()
        data_df.columns = final_columns
        data_df.dropna(how='all', inplace=True)
        
        data_df["Rok"] = year
        data_df["Kraj"] = kraj
        data_df["Sféra"] = sfera
        
        if je_vek and len(final_columns) > 0:
            first_col_name = final_columns[0]
            current_gender = "Neznámé"
            genders = []
            for val in data_df[first_col_name]:
                val_str = str(val).strip().lower()
                if 'celkem' in val_str: current_gender = "Celkem"
                elif 'muži' in val_str or 'muzi' in val_str: current_gender = "Muži"
                elif 'ženy' in val_str or 'zeny' in val_str: current_gender = "Ženy"
                genders.append(current_gender)
                
            data_df["Pohlaví"] = genders
            data_df.rename(columns={first_col_name: "Věk"}, inplace=True)
            data_df = data_df[data_df["Věk"].astype(str).str.contains("let", na=False)]
            
            sloupce_k_pouziti = [c for c in pozadovane_sloupce_vek if c in data_df.columns]
            data_df = data_df[sloupce_k_pouziti]
            vek_rows.append(data_df)
            
        elif je_vzdelani and len(final_columns) > 0:
            first_col_name = final_columns[0]
            data_df.rename(columns={first_col_name: "Vzdělání"}, inplace=True)
            data_df = data_df[~data_df["Vzdělání"].astype(str).str.lower().str.contains("celkem", na=False)]
            
            sloupce_k_pouziti = [c for c in pozadovane_sloupce_vzdelani if c in data_df.columns]
            data_df = data_df[sloupce_k_pouziti]
            vzdelani_rows.append(data_df)
            
    return vek_rows, vzdelani_rows
def parse_all_years():
    all_vek = []
    all_vzdelani = []
    
    for year in range(2011, 2025):
        year_folder = f"{BASE_FOLDER}/{year}"
        if not os.path.exists(year_folder): continue
            
        files = os.listdir(year_folder)
        print(f"Zpracovávám rok {year} (počet souborů: {len(files)})")
        
        for file in files:
            if not file.endswith((".xls", ".xlsx")): continue
            kraj, sfera = extract_metadata(file)
            if not kraj: continue
                
            path = f"{year_folder}/{file}"
            vek_list, vzd_list = parse_excel_file(path, year, kraj, sfera)
            all_vek.extend(vek_list)
            all_vzdelani.extend(vzd_list)
                
    df_vek = pd.concat(all_vek, ignore_index=True) if all_vek else pd.DataFrame()
    df_vzdelani = pd.concat(all_vzdelani, ignore_index=True) if all_vzdelani else pd.DataFrame()
    
    if not df_vek.empty: df_vek.dropna(axis=1, how='all', inplace=True)
    if not df_vzdelani.empty: df_vzdelani.dropna(axis=1, how='all', inplace=True)
    
    return df_vek, df_vzdelani
print("Spouštím parser, může to chvilku trvat...")
df_vek, df_vzdelani = parse_all_years()
print("\n=== HOTOVO ===")
print("VĚK DataFrame velikost:", df_vek.shape)
print("VZDĚLÁNÍ DataFrame velikost:", df_vzdelani.shape)


Spouštím parser, může to chvilku trvat...
Zpracovávám rok 2011 (počet souborů: 30)
Zpracovávám rok 2012 (počet souborů: 30)
Zpracovávám rok 2013 (počet souborů: 30)
Zpracovávám rok 2014 (počet souborů: 30)
Zpracovávám rok 2015 (počet souborů: 30)
Zpracovávám rok 2016 (počet souborů: 30)
Zpracovávám rok 2017 (počet souborů: 30)
Zpracovávám rok 2018 (počet souborů: 30)
Zpracovávám rok 2019 (počet souborů: 30)
Zpracovávám rok 2020 (počet souborů: 30)
Zpracovávám rok 2021 (počet souborů: 30)
Zpracovávám rok 2022 (počet souborů: 30)
Zpracovávám rok 2023 (počet souborů: 30)
Zpracovávám rok 2024 (počet souborů: 30)

=== HOTOVO ===
VĚK DataFrame velikost: (14706, 20)
VZDĚLÁNÍ DataFrame velikost: (526, 19)


Zkoumání:

In [2]:
df_vzdelani["Rok"].unique()


array([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021,
       2022, 2023, 2024])

In [3]:
list(df_vek.columns)

['Rok',
 'Sféra',
 'Pohlaví',
 'Věk',
 'Kraj',
 'Počet zaměstnanců přepočtený podle placených měsíců_tis. osob',
 'Hrubá měsíční mzda_Medián_Kč/měs',
 'Hrubá měsíční mzda_Meziroční index_%',
 'Diferenciace hrubé měsíční mzdy_1. decil_Kč/měs',
 'Diferenciace hrubé měsíční mzdy_1. kvartil_Kč/měs',
 'Diferenciace hrubé měsíční mzdy_3. kvartil_Kč/měs',
 'Diferenciace hrubé měsíční mzdy_9. decil_Kč/měs',
 'Hrubá měsíční mzda_Průměr_Kč/měs',
 'Hrubý měsíční plat_Medián_Kč/měs',
 'Hrubý měsíční plat_Meziroční index_%',
 'Diferenciace hrubého měsíčního platu_1. decil_Kč/měs',
 'Diferenciace hrubého měsíčního platu_1. kvartil_Kč/měs',
 'Diferenciace hrubého měsíčního platu_3. kvartil_Kč/měs',
 'Diferenciace hrubého měsíčního platu_9. decil_Kč/měs',
 'Hrubý měsíční plat_Průměr_Kč/měs']

Odstranění agregovaných řádků:

In [4]:
df_vek = df_vek[df_vek["Pohlaví"] != "Celkem"]

Odstranění prázdných řádků:

In [5]:
df_vzdelani = df_vzdelani[
    df_vzdelani["Vzdělání"].isin([
        "Základní a nedokončené",
        "Střední bez maturity",
        "Střední s maturitou",
        "Vyšší odborné a bakalářské",
        "Vysokoškolské"
    ])
    & df_vzdelani["Vzdělání"].notna()
]

Sjednocení hodnot mzdy a platu do jednoho sloupce:

In [6]:
def sjednot_mzdu_a_plat(df_vek):
    sloupce = {
        "Plat_Medián": (
            "Hrubý měsíční plat_Medián_Kč/měs",
            "Hrubá měsíční mzda_Medián_Kč/měs"
        ),
        "Plat_Průměr": (
            "Hrubý měsíční plat_Průměr_Kč/měs",
            "Hrubá měsíční mzda_Průměr_Kč/měs"
        ),
        "Plat_P10": (
            "Diferenciace hrubého měsíčního platu_1. decil_Kč/měs",
            "Diferenciace hrubé měsíční mzdy_1. decil_Kč/měs"
        ),
        "Plat_Q1": (
            "Diferenciace hrubého měsíčního platu_1. kvartil_Kč/měs",
            "Diferenciace hrubé měsíční mzdy_1. kvartil_Kč/měs"
        ),
        "Plat_Q3": (
            "Diferenciace hrubého měsíčního platu_3. kvartil_Kč/měs",
            "Diferenciace hrubé měsíční mzdy_3. kvartil_Kč/měs"
        ),
        "Plat_P90": (
            "Diferenciace hrubého měsíčního platu_9. decil_Kč/měs",
            "Diferenciace hrubé měsíční mzdy_9. decil_Kč/měs"
        ),
        "Plat_Index": (
            "Hrubý měsíční plat_Meziroční index_%",
            "Hrubá měsíční mzda_Meziroční index_%"
        ),
    }

    for new_col, (plat_col, mzda_col) in sloupce.items():
        s_plat = df_vek[plat_col] if plat_col in df_vek.columns else None
        s_mzda = df_vek[mzda_col] if mzda_col in df_vek.columns else None

        if s_plat is not None and s_mzda is not None:
            df_vek[new_col] = s_plat.combine_first(s_mzda)
        elif s_plat is not None:
            df_vek[new_col] = s_plat
        elif s_mzda is not None:
            df_vek[new_col] = s_mzda

    k_odstraneni = []
    for plat_col, mzda_col in sloupce.values():
        if plat_col in df_vek.columns:
            k_odstraneni.append(plat_col)
        if mzda_col in df_vek.columns:
            k_odstraneni.append(mzda_col)

    df_vek = df_vek.drop(columns=k_odstraneni)

    return df_vek



df_vek = sjednot_mzdu_a_plat(df_vek)

df_vzdelani = sjednot_mzdu_a_plat(df_vzdelani)


Odstranění prázdných řádků:

In [7]:
df_vek = df_vek[
    df_vek["Plat_Průměr"]
    .astype(str)
    .str.strip()
    .replace("nan", "")
    != ""
]


Odstranění prázdných řádků z df_vek a hodnot pro celou ČR

In [8]:
# Ponecháme pouze řádky, kde Věk NENÍ "do 20 let" a zároveň Kraj NENÍ "Česká republika"
df_vek = df_vek[(df_vek['Věk'] != 'do 20 let') & (df_vek['Kraj'] != 'Česká republika')]

Odstranění nepotřebných sloupců z df_vzdelani

In [10]:
df_vzdelani.drop("Kraj", axis=1, inplace=True)
df_vzdelani.drop("Plat_Index", axis=1, inplace=True)

Uložení do Excelu

In [11]:
# Cesta, kam se mají finální Excely uložit
output_dir = r"C:\Users\marie\Documents\Datová analytika - Python, Big data, ML\Projekt"

print("Ukládám df_vek do Excelu...")
# index=False zaručí, že se do Excelu nebude ukládat zbytečný pandas index (0, 1, 2...)
df_vek.to_excel(fr"{output_dir}\data_vek.xlsx", index=False)

print("Ukládám df_vzdelani do Excelu...")
df_vzdelani.to_excel(fr"{output_dir}\data_vzdelani.xlsx", index=False)

print("Uloženo úspěšně!")


Ukládám df_vek do Excelu...
Ukládám df_vzdelani do Excelu...
Uloženo úspěšně!


Stažení inflace:

In [ ]:
!pip install PyPDF2

import urllib.request
import re
import pandas as pd
import PyPDF2
import io

def main():
    url = 'https://csu.gov.cz/docs/107508/a3bcc692-1894-b309-99d1-470e95b65144/inflace_2000_2025.pdf'
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    response = urllib.request.urlopen(req)
    pdf_file = io.BytesIO(response.read())

    reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"

    # Match lines like "v roce 2000.............................3,9 %"
    pattern = r"v\s+roce\s+(\d{4})\D+([\d,]+)\s*%"
    matches = re.findall(pattern, text)

    data = []
    for year, rate in matches:
        data.append({
            'rok': int(year),
            'míra inflace v procentech': float(rate.replace(',', '.'))
        })

    if not data:
        print("No data extracted. Here is the raw text to debug:")
        print(text)
        return

    df = pd.DataFrame(data)
    # Sort by year just in case
    df = df.sort_values(by='rok')
    
    output_file = 'inflace_2000_2025.xlsx'
    df.to_excel(output_file, index=False)
    print(f"Done! Saved {len(df)} rows to {output_file}")

if __name__ == '__main__':
    main()


ModuleNotFoundError: No module named 'PyPDF2'